## SelfGNN — Yelp Feature Extraction (8+8 Schema)

**Prerequisite:** Run `preprocess_yelp.ipynb` first.

| Side | Dim | Features |
|------|-----|----------|
| User | 8 | `interaction_count`, `avg_interaction_value`, `unique_merchant_count`, `activity_span_days`, `recency_score`, `txns_per_week`, `value_std_norm`, `repeat_visit_rate` |
| Merchant | 8 | `interaction_count`, `avg_interaction_value`, `unique_user_count`, `category_id` (MCC index), `txns_per_user`, `value_std_norm`, `popularity_rank`, `user_repeat_rate` |
| Edge | 1 | `normalized_interaction_value` (stars / 5.0 → [0.2, 1.0]) |

**Outputs:** `Datasets/yelp-merchant/`
- `user_features.npy` — shape (N_users, 8)
- `merchant_features.npy` — shape (N_merchants, 8)
- `edge_weights.pkl`, `feature_meta.json`, `bipartite_graph_sample.png`

In [1]:
# ── Configuration ──────────────────────────────────────────────────────────────
OUT_DIR      = '../Datasets/yelp-merchant/'
RAW_DIR      = '../datasetRaw/yelp/'
DATASET_NAME = 'yelp-merchant'

import os, json, pickle
import numpy as np
import pandas as pd
from collections import defaultdict
from datetime import datetime

with open(OUT_DIR + 'user2id.pkl', 'rb') as f:
    user2id = pickle.load(f)
with open(OUT_DIR + 'merchant2id.pkl', 'rb') as f:
    merchant2id = pickle.load(f)

usrnum      = len(user2id)
merchantnum = len(merchant2id)
print(f'Loaded mappings: {usrnum:,} users, {merchantnum:,} merchants')

Loaded mappings: 268,653 users, 109,325 merchants


In [2]:
# ── Cell 2: Build merchant MCC mapping (consistent with Finance/Synthetic) ─────
# Yelp category strings → ISO 18245 MCC codes used by Finance/Synthetic datasets.
# Walk ALL categories per business; assign the first matched MCC (most-specific first).
# Fallback: MCC 5999 (Miscellaneous Retail) for unmapped categories.
YELP_TO_MCC = {
    # Restaurants / Food Service
    'Restaurants': 5812, 'Pizza': 5812, 'Chinese': 5812, 'Mexican': 5812,
    'Italian': 5812, 'Sushi Bars': 5812, 'Thai': 5812, 'Japanese': 5812,
    'American (Traditional)': 5812, 'American (New)': 5812, 'Indian': 5812,
    'Mediterranean': 5812, 'Greek': 5812, 'Vietnamese': 5812, 'Korean': 5812,
    'Sandwiches': 5812, 'Salad': 5812, 'Seafood': 5812, 'Steakhouses': 5812,
    'Breakfast & Brunch': 5812, 'Food': 5812, 'Juice Bars & Smoothies': 5812,
    'Desserts': 5812, 'Ice Cream & Frozen Yogurt': 5812, 'Diners': 5812,
    'Cafes': 5812, 'Soup': 5812, 'Noodles': 5812, 'Caterers': 5812,
    # Fast Food
    'Burgers': 5814, 'Fast Food': 5814,
    # Bakeries
    'Bakeries': 5462, 'Bagels': 5462, 'Donuts': 5462,
    # Coffee
    'Coffee & Tea': 5814,
    # Candy / Confectionery
    'Candy Stores': 5441, 'Chocolate': 5441,
    # Bars / Nightlife
    'Bars': 5813, 'Nightlife': 5813, 'Beer Bar': 5813, 'Wine Bars': 5813,
    'Cocktail Bars': 5813, 'Pubs': 5813, 'Sports Bars': 5813,
    'Dance Clubs': 5813, 'Karaoke': 5813, 'Jazz & Blues': 5813,
    # Grocery / Supermarkets
    'Grocery': 5411, 'Supermarkets': 5411, 'Farmers Market': 5411,
    'Health Markets': 5411, 'Organic Stores': 5411, 'Convenience Stores': 5411,
    # Specialty Food
    'Specialty Food': 5422, 'Cheese Shops': 5422, 'Meat Shops': 5422,
    # Drug Stores / Pharmacies
    'Drugstores': 5912, 'Pharmacy': 5912,
    # General Shopping / Gifts
    'Shopping': 5999, 'Gift Shops': 5947, 'Antiques': 5932,
    'Thrift Stores': 5932, 'Flea Markets': 5999,
    # Clothing
    'Fashion': 5691, "Women's Clothing": 5621, "Men's Clothing": 5611,
    "Children's Clothing": 5641,
    # Shoes / Jewelry
    'Shoe Stores': 5661, 'Jewelry': 5094,
    # Books / Sporting / Electronics
    'Bookstores': 5942, 'Sporting Goods': 5941, 'Outdoor Gear': 5941,
    'Electronics': 5732, 'Computers': 5732, 'Mobile Phones': 5732,
    # Department / Home / Furniture
    'Department Stores': 5311,
    'Home & Garden': 5251, 'Hardware Stores': 5251, 'Nurseries & Gardening': 5251,
    'Furniture Stores': 5712, 'Home Decor': 5712,
    # Florists / Toys / Pets
    'Florists': 5992, 'Toy Stores': 5945,
    'Pet Stores': 5995, 'Pet Food Stores': 5995,
    # Automotive
    'Automotive': 5533, 'Auto Parts & Supplies': 5533, 'Gas Stations': 5541,
    'Car Dealers': 5511, 'Auto Repair': 7538, 'Body Shops': 7531,
    'Car Wash': 7542, 'Tires': 5571,
    # Hotels / Travel
    'Hotels & Travel': 7011, 'Hotels': 7011, 'Bed & Breakfast': 7011,
    'Hostels': 7011, 'Vacation Rentals': 7011, 'Travel Agents': 4722,
    # Entertainment
    'Arts & Entertainment': 7922, 'Movie Theaters': 7832,
    'Amusement Parks': 7996, 'Music Venues': 7929, 'Museums': 7999,
    'Performing Arts': 7922, 'Theaters': 7922, 'Comedy Clubs': 7929,
    'Bowling': 7933, 'Golf': 7992, 'Arcades': 7993, 'Escape Games': 7999,
    # Fitness / Sports / Parks
    'Gyms': 7941, 'Fitness & Instruction': 7941, 'Yoga': 7941,
    'Pilates': 7941, 'Martial Arts': 7941, 'Swimming Pools': 7997,
    'Tennis': 7941, 'Cycling': 7941,
    'Parks': 7999, 'Hiking': 7999, 'Beaches': 7999,
    # Beauty / Hair
    'Beauty & Spas': 7299, 'Hair Salons': 7230, 'Nail Salons': 7230,
    'Spas': 7299, 'Skin Care': 7299, 'Waxing': 7299, 'Tanning': 7299,
    'Barbers': 7230, 'Makeup Artists': 7299, 'Eyelash Service': 7299,
    # Health / Medical
    'Health & Medical': 8099, 'Doctors': 8011, 'Dentists': 8021,
    'Optometrists': 8042, 'Hospitals': 8062, 'Chiropractors': 8099,
    'Physical Therapy': 8099, 'Mental Health': 8049,
    'Nutritionists': 8099, 'Acupuncture': 8099, 'Massage': 8099,
    # Veterinary
    'Veterinarians': 742, 'Pet Services': 742, 'Animal Shelters': 742,
    # Financial
    'Financial Services': 6099, 'Banks & Credit Unions': 6012,
    'Insurance': 6411, 'Check Cashing/Pay-day Loans': 6099,
    # Real Estate
    'Real Estate': 6512, 'Property Management': 6512,
    # Professional / IT
    'Professional Services': 7372, 'IT Services & Computer Repair': 7372,
    'Lawyers': 8111, 'Accountants': 8931, 'Marketing': 7372,
    'Graphic Design': 7372, 'Photographers': 7372, 'Web Design': 7372,
    # Printing / Shipping
    'Printing Services': 2759, 'Shipping Centers': 4215,
    # Education
    'Education': 8299, 'Tutoring Centers': 8299, 'Preschools': 8299,
    'Elementary Schools': 8211, 'Middle Schools & High Schools': 8211,
    'Colleges & Universities': 8220, 'Driving Schools': 8299,
    'Language Schools': 8299, 'Music & Instruction': 8299,
    # Home Services
    'Home Services': 7349, 'Plumbing': 1711, 'Electricians': 1731,
    'Contractors': 1520, 'Roofing': 1761, 'Landscaping': 780,
    'Cleaning': 7349, 'Carpet Cleaning': 7217, 'HVAC': 1711,
    'Movers': 4214, 'Handyman': 7549,
    # Event Planning / Entertainment Services
    'Event Planning & Services': 7999, 'Wedding Planning': 7999,
    'Party & Event Planning': 7999, 'DJs': 7929,
    # Laundry
    'Dry Cleaning': 7216, 'Laundry Services': 7211,
    # Transport / Parking
    'Transportation': 4111, 'Taxis': 4121, 'Parking': 7523,
    # Religious / Government
    'Religious Organizations': 8661, 'Public Services & Government': 9399,
}
MCC_UNKNOWN = 5999  # Miscellaneous Retail — fallback for unmapped categories

bid_to_mcc: dict = {}
business_path = RAW_DIR + 'yelp_academic_dataset_business.json'

with open(business_path, 'r', encoding='utf-8') as f:
    for line in f:
        b   = json.loads(line.strip())
        bid = b.get('business_id', '')
        if bid not in merchant2id:
            continue
        cats       = b.get('categories') or ''
        cat_tokens = [c.strip() for c in cats.split(',') if c.strip()]
        # Walk all listed categories; assign first MCC match
        mcc = MCC_UNKNOWN
        for cat in cat_tokens:
            if cat in YELP_TO_MCC:
                mcc = YELP_TO_MCC[cat]
                break
        bid_to_mcc[bid] = mcc

unique_mccs = sorted(set(bid_to_mcc.values()))
mcc2idx     = {mcc: i for i, mcc in enumerate(unique_mccs)}
print(f'Unique MCC codes mapped: {len(unique_mccs):,}  |  codes: {unique_mccs[:20]}')

merchant_cat = np.zeros(merchantnum, dtype=np.float32)
for bid, mcc in bid_to_mcc.items():
    mid = merchant2id.get(bid)
    if mid is not None:
        merchant_cat[mid] = float(mcc2idx[mcc])

Unique MCC codes mapped: 80  |  codes: [742, 780, 1520, 1711, 1731, 1761, 2759, 4111, 4121, 4214, 4215, 4722, 5094, 5251, 5311, 5411, 5422, 5441, 5462, 5511]


In [3]:
# ── Cell 3: Stream reviews → build per-user/merchant/edge statistics ───────────
def parse_ts(s: str):
    try:
        return datetime.strptime(s, '%Y-%m-%d %H:%M:%S').timestamp()
    except Exception:
        return None

u_count     = np.zeros(usrnum,      dtype=np.int64)
u_val_sum   = np.zeros(usrnum,      dtype=np.float64)
u_val_sq    = np.zeros(usrnum,      dtype=np.float64)
u_merchants = [set() for _ in range(usrnum)]
u_min_ts    = np.full(usrnum,  np.inf)
u_max_ts    = np.full(usrnum, -np.inf)

m_count     = np.zeros(merchantnum, dtype=np.int64)
m_val_sum   = np.zeros(merchantnum, dtype=np.float64)
m_val_sq    = np.zeros(merchantnum, dtype=np.float64)
m_users     = [set() for _ in range(merchantnum)]

edge_val_sum: dict = defaultdict(float)
edge_cnt:     dict = defaultdict(int)
global_max_ts      = -np.inf

n_kept = 0
review_path = RAW_DIR + 'yelp_academic_dataset_review.json'
with open(review_path, 'r', encoding='utf-8') as f:
    for line in f:
        rv      = json.loads(line.strip())
        uid_str = rv.get('user_id', '')
        bid     = rv.get('business_id', '')
        stars   = rv.get('stars')
        if uid_str not in user2id or bid not in merchant2id or stars is None:
            continue
        uid  = user2id[uid_str]
        mid  = merchant2id[bid]
        norm = float(stars) / 5.0

        u_count[uid]   += 1
        u_val_sum[uid]  += norm
        u_val_sq[uid]   += norm * norm
        u_merchants[uid].add(mid)
        m_count[mid]   += 1
        m_val_sum[mid]  += norm
        m_val_sq[mid]   += norm * norm
        m_users[mid].add(uid)

        ts = parse_ts(rv.get('date', ''))
        if ts is not None:
            if ts < u_min_ts[uid]: u_min_ts[uid] = ts
            if ts > u_max_ts[uid]: u_max_ts[uid] = ts
            if ts > global_max_ts: global_max_ts = ts

        edge_val_sum[(uid, mid)] += norm
        edge_cnt[(uid, mid)]     += 1
        n_kept += 1

print(f'Processed {n_kept:,} reviews')

Processed 4,190,201 reviews


In [4]:
# ── Cell 4: Build 8+8 feature arrays ──────────────────────────────────────────
SECS_PER_DAY  = 86400.0
SECS_PER_YEAR = 365.0 * SECS_PER_DAY

u_avg_val       = np.where(u_count > 0, u_val_sum / u_count, 0.0).astype(np.float32)
u_var           = np.maximum(u_val_sq / np.maximum(u_count, 1) - (u_val_sum / np.maximum(u_count, 1))**2, 0.0)
u_val_std       = np.sqrt(u_var).astype(np.float32)
u_unique_merch  = np.array([len(s) for s in u_merchants], dtype=np.float32)
u_span_days     = np.where(np.isfinite(u_min_ts) & np.isfinite(u_max_ts),
                            (u_max_ts - u_min_ts) / SECS_PER_DAY, 0.0).astype(np.float32)
u_recency       = np.where(np.isfinite(u_max_ts),
                            np.exp(-np.maximum(0.0, global_max_ts - u_max_ts) / SECS_PER_YEAR),
                            0.0).astype(np.float32)
u_txns_per_week = (u_count.astype(np.float32) * 7.0 / np.maximum(u_span_days, 1.0))
u_repeat_rate   = (u_count.astype(np.float32) / np.maximum(u_unique_merch, 1.0))

m_avg_val       = np.where(m_count > 0, m_val_sum / m_count, 0.0).astype(np.float32)
m_var           = np.maximum(m_val_sq / np.maximum(m_count, 1) - (m_val_sum / np.maximum(m_count, 1))**2, 0.0)
m_val_std       = np.sqrt(m_var).astype(np.float32)
m_unique_users  = np.array([len(s) for s in m_users], dtype=np.float32)
m_txns_per_u    = (m_count.astype(np.float32) / np.maximum(m_unique_users, 1.0))

m_repeat_u = np.zeros(merchantnum, dtype=np.float32)
for (uid, mid), cnt in edge_cnt.items():
    if cnt > 1:
        m_repeat_u[mid] += 1.0
m_user_repeat_rate = m_repeat_u / np.maximum(m_unique_users, 1.0)

m_count_order = np.argsort(m_count)
m_pop_rank    = np.zeros(merchantnum, dtype=np.float32)
m_pop_rank[m_count_order] = np.linspace(0.0, 1.0, merchantnum)

user_features = np.stack([
    u_count.astype(np.float32),  # 0: interaction_count
    u_avg_val,                    # 1: avg_interaction_value (stars/5)
    u_unique_merch,               # 2: unique_merchant_count
    u_span_days,                  # 3: activity_span_days
    u_recency,                    # 4: recency_score
    u_txns_per_week,              # 5: txns_per_week
    u_val_std,                    # 6: value_std_norm
    u_repeat_rate,                # 7: repeat_visit_rate
], axis=1)

merchant_features = np.stack([
    m_count.astype(np.float32),  # 0: interaction_count
    m_avg_val,                    # 1: avg_interaction_value
    m_unique_users,               # 2: unique_user_count
    merchant_cat,                 # 3: category_id (MCC index)
    m_txns_per_u,                 # 4: txns_per_user
    m_val_std,                    # 5: value_std_norm
    m_pop_rank,                   # 6: popularity_rank
    m_user_repeat_rate,           # 7: user_repeat_rate
], axis=1)

print(f'User features     : {user_features.shape}')
print(f'Merchant features : {merchant_features.shape}')
print(f'User   mean/col   : {user_features.mean(axis=0).round(4)}')
print(f'Merch  mean/col   : {merchant_features.mean(axis=0).round(4)}')

User features     : (268653, 8)
Merchant features : (109325, 8)
User   mean/col   : [1.5597100e+01 7.6400000e-01 1.4887500e+01 1.3244045e+03 2.4560000e-01
 2.0441000e+00 2.1910000e-01 1.0325000e+00]
Merch  mean/col   : [38.3279  0.7397 36.5843 36.9017  1.0466  0.2288  0.5     0.0403]


In [5]:
# ── Cell 5: Normalize features ─────────────────────────────────────────────────
# User  skip: col 1 (avg_value in [0.2,1]), col 4 (recency in [0,1])
# User  minmax-only: col 6 (value_std)
# Merch skip: col 1 (avg_value), col 6 (pop_rank in [0,1]), col 7 (repeat_rate in [0,1])
# Merch minmax-only: col 5 (value_std)

SKIP_NORM_USER  = {1, 4}
SKIP_NORM_MERCH = {1, 6, 7}

def log_minmax(arr: np.ndarray) -> np.ndarray:
    v = np.log1p(np.clip(arr, 0, None))
    mn, mx = v.min(), v.max()
    return ((v - mn) / max(mx - mn, 1e-8)).astype(np.float32)

def minmax(arr: np.ndarray) -> np.ndarray:
    mn, mx = arr.min(), arr.max()
    return ((arr - mn) / max(mx - mn, 1e-8)).astype(np.float32)

user_features_norm  = user_features.copy()
merch_features_norm = merchant_features.copy()

for col in range(8):
    if col not in SKIP_NORM_USER:
        user_features_norm[:, col] = (
            minmax(user_features[:, col]) if col == 6
            else log_minmax(user_features[:, col])
        )
for col in range(8):
    if col not in SKIP_NORM_MERCH:
        merch_features_norm[:, col] = (
            minmax(merchant_features[:, col]) if col == 5
            else log_minmax(merchant_features[:, col])
        )

print('After normalization:')
print(f'  User   min/max per col: {user_features_norm.min(axis=0).round(3)} / {user_features_norm.max(axis=0).round(3)}')
print(f'  Merch  min/max per col: {merch_features_norm.min(axis=0).round(3)} / {merch_features_norm.max(axis=0).round(3)}')

After normalization:
  User   min/max per col: [0.  0.2 0.  0.  0.  0.  0.  0. ] / [1. 1. 1. 1. 1. 1. 1. 1.]
  Merch  min/max per col: [0.  0.2 0.  0.  0.  0.  0.  0. ] / [1.  1.  1.  1.  1.  1.  1.  0.8]


In [6]:
# ── Cell 6: Edge weights dict ──────────────────────────────────────────────────
edge_weights = {
    (uid, mid): float(edge_val_sum[(uid, mid)] / edge_cnt[(uid, mid)])
    for (uid, mid) in edge_cnt
}
vals = np.array(list(edge_weights.values()))
print(f'Edge weights: {len(edge_weights):,} pairs')
print(f'  min={vals.min():.4f}, max={vals.max():.4f}, mean={vals.mean():.4f}')

Edge weights: 3,999,575 pairs
  min=0.2000, max=1.0000, mean=0.7682


In [7]:
# ── Cell 7: Bipartite graph visualization ──────────────────────────────────────
# Geometric bipartite layout — users (left, blue circles) ↔ merchants (right, orange squares).
# Nodes are the top-N highest-degree users and their connected merchants.
# Edge thickness ∝ interaction weight (normalized to [0,1]).

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from collections import defaultdict as _dd

    np.random.seed(42)
    N_USR_SHOW = 20
    N_MRC_SHOW = 25

    # Index edge_weights by user
    u_to_m: dict = _dd(list)
    for (uid, mid), w in edge_weights.items():
        u_to_m[uid].append((mid, w))

    # Select highest-degree users
    top_users  = sorted(u_to_m.keys(), key=lambda u: len(u_to_m[u]), reverse=True)
    sample_u   = top_users[:N_USR_SHOW]

    # Gather connected merchants, capped at N_MRC_SHOW
    sample_m_set: set = set()
    for uid in sample_u:
        for mid, _ in u_to_m[uid]:
            sample_m_set.add(mid)
    sample_m    = sorted(sample_m_set)[:N_MRC_SHOW]
    m_visible   = set(sample_m)

    n_u, n_m = len(sample_u), len(sample_m)

    # Bipartite layout: users x=0, merchants x=1
    u_pos = {uid: (0.0, i / max(n_u - 1, 1)) for i, uid in enumerate(sample_u)}
    m_pos = {mid: (1.0, i / max(n_m - 1, 1)) for i, mid in enumerate(sample_m)}

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.axis('off')
    ax.set_title(
        f'Bipartite Interaction Graph — {DATASET_NAME}\n'
        f'(top-{n_u} users by degree  ·  {n_m} connected merchants)',
        fontsize=13, fontweight='bold', pad=14,
    )

    visible_ws = [w for uid in sample_u for mid, w in u_to_m[uid] if mid in m_visible]
    w_max = max(visible_ws) if visible_ws else 1.0

    # Draw edges
    for uid in sample_u:
        for mid, w in u_to_m[uid]:
            if mid not in m_visible:
                continue
            p1, p2 = u_pos[uid], m_pos[mid]
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]],
                    color='#90A4AE', alpha=0.4,
                    linewidth=0.4 + 2.2 * (w / w_max), zorder=1)

    # User nodes (blue circles)
    ax.scatter([u_pos[u][0] for u in sample_u], [u_pos[u][1] for u in sample_u],
               s=220, c='#1565C0', zorder=4, edgecolors='white', linewidths=1.8, label='User')
    for uid, (x, y) in u_pos.items():
        ax.text(x - 0.05, y, f'u{uid}', ha='right', va='center', fontsize=7, color='#1565C0')

    # Merchant nodes (orange squares)
    ax.scatter([m_pos[m][0] for m in sample_m], [m_pos[m][1] for m in sample_m],
               s=220, c='#E65100', marker='s', zorder=4, edgecolors='white', linewidths=1.8, label='Merchant')
    for mid, (x, y) in m_pos.items():
        ax.text(x + 0.05, y, f'm{mid}', ha='left', va='center', fontsize=7, color='#E65100')

    ax.text(0.0, -0.08, f'Users  (n={n_u})',
            ha='center', fontsize=11, color='#1565C0', fontweight='bold',
            transform=ax.transData)
    ax.text(1.0, -0.08, f'Merchants  (n={n_m})',
            ha='center', fontsize=11, color='#E65100', fontweight='bold',
            transform=ax.transData)

    ax.set_xlim(-0.22, 1.22)
    ax.set_ylim(-0.12, 1.08)
    ax.legend(loc='upper center', ncol=2, fontsize=9, framealpha=0.8)

    plt.tight_layout()
    plt.savefig(OUT_DIR + 'bipartite_graph_sample.png', bbox_inches='tight', dpi=120)
    plt.show()
    print(f'Saved: bipartite_graph_sample.png  ({n_u} users, {n_m} merchants, {len(visible_ws)} edges shown)')

except Exception as e:
    print(f'Graph visualization skipped: {e}')

Saved: bipartite_graph_sample.png  (20 users, 25 merchants, 29 edges shown)


/var/folders/sr/2htv7_wj6sj6qtmjtk9f1vpc0000gn/T/ipykernel_27887/2872773007.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ── Cell 8: Save all files + summary ───────────────────────────────────────────
np.save(OUT_DIR + 'user_features.npy',     user_features_norm)
np.save(OUT_DIR + 'merchant_features.npy', merch_features_norm)

with open(OUT_DIR + 'edge_weights.pkl', 'wb') as f:
    pickle.dump(edge_weights, f)

meta = {
    'user_feature_names': [
        'interaction_count', 'avg_interaction_value', 'unique_merchant_count',
        'activity_span_days', 'recency_score', 'txns_per_week',
        'value_std_norm', 'repeat_visit_rate',
    ],
    'merchant_feature_names': [
        'interaction_count', 'avg_interaction_value', 'unique_user_count',
        'category_id', 'txns_per_user', 'value_std_norm',
        'popularity_rank', 'user_repeat_rate',
    ],
    'edge_feature_names':  ['normalized_interaction_value'],
    'user_value_col':      1,
    'merchant_value_col':  1,
    'edge_normalization':  'stars / 5.0',
    'dataset':             DATASET_NAME,
    'schema':              '8+8',
    'mcc_unique_count':    len(unique_mccs),
    'category_encoding':   'Yelp-category -> MCC code -> contiguous int index (mcc2idx)',
}
with open(OUT_DIR + 'feature_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved:')
for fname in ['user_features.npy', 'merchant_features.npy', 'edge_weights.pkl', 'feature_meta.json', 'bipartite_graph_sample.png']:
    path = OUT_DIR + fname
    size = os.path.getsize(path) if os.path.isfile(path) else None
    status = f'{size/1024:.0f} KB' if size else 'MISSING'
    print(f'  {fname:<35} {status}')
print()
print('=' * 60)
print(f'Feature Extraction Complete — {DATASET_NAME}')
print('=' * 60)
print(f'User feature shape     : {user_features_norm.shape}')
print(f'Merchant feature shape : {merch_features_norm.shape}')
print(f'Edge weights           : {len(edge_weights):,} pairs')
print(f'Schema (8+8)           : user_dim={user_features_norm.shape[1]}, merch_dim={merch_features_norm.shape[1]}')

Saved:
  user_features.npy                   8396 KB
  merchant_features.npy               3417 KB
  edge_weights.pkl                    75380 KB
  feature_meta.json                   1 KB
  bipartite_graph_sample.png          196 KB

Feature Extraction Complete — yelp-merchant
User feature shape     : (268653, 8)
Merchant feature shape : (109325, 8)
Edge weights           : 3,999,575 pairs
Schema (8+8)           : user_dim=8, merch_dim=8
